# Data Cleaning & Bombarding
**Input:** `data/raw/globalterrorismdb_0718dist.csv`  
**Goal:** Select relevant columns, rename them, clean the data, and save the processed output.

### Imports
Loads `pandas` for data manipulation and suppresses non-critical warnings.

In [12]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## 1. Load Extracted Data

### Read Processed CSV
Loads the extracted dataset saved from Notebook 1. Uses `latin-1` encoding and `low_memory=False` to handle the large file without type inference warnings.

In [13]:
df = pd.read_csv('../data/raw/globalterrorismdb_0718dist.csv', encoding='latin-1', low_memory=False)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

Loaded: 181,691 rows × 135 columns


## 2. Select & Rename Columns

### Column Selection
Keeps only the 9 relevant columns from the 135 available and renames them to cleaner, more readable names:
- `iyear` → `Year`
- `imonth` → `Month`
- `extended` → `Duration` (1 = attack lasted >24 hrs, 0 = ≤24 hrs)
- `country_txt` → `Country`
- `region_txt` → `Region`
- `city` → `City`
- `crit1` → `Crime`
- `motive` → `Reason`
- `individual` → `Individual`

In [14]:
COLUMNS = {
    'iyear'       : 'Year',
    'imonth'      : 'Month',
    'extended'    : 'Duration',
    'country_txt' : 'Country',
    'region_txt'  : 'Region',
    'city'        : 'City',
    'crit1'       : 'Crime',
    'motive'      : 'Reason',
    'individual'  : 'Individual',
}

df = df[list(COLUMNS.keys())].rename(columns=COLUMNS)
print(f'Shape after selection: {df.shape}')
df.head()

Shape after selection: (181691, 9)


,Year,Month,Duration,Country,Region,City,Crime,Reason,Individual
0,1970,7,0,Dominican Republic,Central America & Caribbean,Santo Domingo,1,NaN,0
1,1970,0,0,Mexico,North America,Mexico city,1,NaN,0
2,1970,1,0,Philippines,Southeast Asia,Unknown,1,NaN,0
3,1970,1,0,Greece,Western Europe,Athens,1,NaN,0
4,1970,1,0,Japan,East Asia,Fukouka,1,NaN,0


## 3. Preprocessing

### Missing Values Before Cleaning
Shows the count and percentage of nulls per column before any cleaning is applied, so we can track what gets removed.

In [15]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})

,missing_count,missing_%
Year,0,0.00
Month,0,0.00
Duration,0,0.00
Country,0,0.00
Region,0,0.00
City,435,0.24
Crime,0,0.00
Reason,131130,72.17
Individual,0,0.00


### Replace Empty Strings & Unknown Placeholders
Replaces empty strings across all columns and `'Unknown'` in `City` and `Reason` with `NaN` so they are treated as missing values and dropped in the next step.

In [16]:
# Replace empty strings and 'Unknown' placeholders with NaN
df.replace('', pd.NA, inplace=True)
df['City'] = df['City'].str.strip().replace('Unknown', pd.NA)
df['Reason'] = df['Reason'].str.strip().replace('Unknown', pd.NA)

### Drop All Rows with NA Values
Drops every row that contains at least one `NaN` across any column. Ensures the final dataset is fully complete with no missing values.

In [17]:
before = len(df)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Dropped {before - len(df):,} rows → {len(df):,} remaining')

Dropped 147,423 rows → 34,268 remaining


## 4. Post-Cleaning Validation

### Missing Values After Cleaning
Re-checks nulls across all columns after dropping. All values should be 0.

In [18]:
df.isnull().sum()

Year          0
Month         0
Duration      0
Country       0
Region        0
City          0
Crime         0
Reason        0
Individual    0
dtype: int64

### Final Shape & Preview
Confirms the final dimensions of the cleaned DataFrame and shows the first few rows.

In [19]:
print(f'Final shape: {df.shape}')
df.head()

Final shape: (34268, 9)


,Year,Month,Duration,Country,Region,City,Crime,Reason,Individual
0,1970,1,0,United States,North America,Cairo,1,To protest the Cairo Illinois Police Deparment,0
1,1970,1,0,United States,North America,Madison,1,To protest the War in Vietnam and the draft,0
2,1970,1,0,United States,North America,Madison,1,To protest the War in Vietnam and the draft,0
3,1970,1,0,United States,North America,Denver,1,Protest the draft and Vietnam War,0
4,1970,1,0,United States,North America,Rio Piedras,1,To protest United States owned businesses in P...,0


### Data Types Summary
Confirms the dtype of each column after cleaning to ensure numeric and categorical columns are correctly typed.

In [20]:
df.dtypes

Year          int64
Month         int64
Duration      int64
Country         str
Region          str
City            str
Crime         int64
Reason          str
Individual    int64
dtype: object

## 5. Save Cleaned Data

### Export to Processed Folder
Saves the cleaned and preprocessed DataFrame to `data/processed/gtd_cleaned.csv`. This file will be used as the input for all subsequent analysis and visualisation notebooks.

In [21]:
OUT_PATH = '../data/processed/gtd_cleaned.csv'
df.to_csv(OUT_PATH, index=False)
print(f'Saved to {OUT_PATH}')

Saved to ../data/processed/gtd_cleaned.csv
